# manifold GPU compute rework — Colab smoke test

Runs the `feature/gpu-compute-rework` branch on a free T4. Validates:

1. WGSL IEEE 754 fp64 library (`fp64_lib.wgsl`) — precision vs CPU fp64
2. Dawn HAL and WGSL sort/BVH/collision kernels produce byte-identical output
3. Whether the experimental GPU fp64 `Kernel12` path performs better on NVIDIA/Vulkan than it does on Apple Silicon/Metal

**Runtime → Change runtime type → T4 GPU** before running.

## Sanity check the runtime

In [ ]:
!nvidia-smi
!echo '---'
!cat /etc/os-release | head -3

## Install build deps (~2–3 min)

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq cmake ninja-build python3 python3-distutils \
  libvulkan-dev vulkan-tools mesa-vulkan-drivers libtbb-dev \
  libxrandr-dev libxinerama-dev libxcursor-dev libxi-dev libx11-dev
echo 'cmake:' $(cmake --version | head -1)
vulkaninfo --summary 2>/dev/null | head -15 || echo 'vulkaninfo not available'

## Clone the branch + Dawn submodule

In [ ]:
%%bash
cd /content
rm -rf manifold
git clone --depth 1 -b feature/gpu-compute-rework \
  https://github.com/DatanoiseTV/manifold.git
cd manifold
git submodule update --init --recursive --depth 1
echo 'Dawn submodule size:' $(du -sh third_party/dawn | cut -f1)

## Configure — Vulkan backend, Metal off

In [ ]:
%%bash
cd /content/manifold
cmake -B build -G Ninja \
  -DMANIFOLD_GPU_WEBGPU=ON \
  -DMANIFOLD_PAR=ON \
  -DMANIFOLD_TEST=OFF \
  -DMANIFOLD_CROSS_SECTION=OFF \
  -DMANIFOLD_PYBIND=OFF \
  -DCMAKE_BUILD_TYPE=Release \
  -DBUILD_SHARED_LIBS=OFF \
  -DDAWN_ENABLE_VULKAN=ON \
  -DDAWN_ENABLE_METAL=OFF \
  -DDAWN_ENABLE_D3D12=OFF \
  2>&1 | tail -15

## Build — this is the long one (~20–30 min on Colab's 2 vCPUs)

In [ ]:
%%bash
cd /content/manifold
cmake --build build -j4 --target manifold 2>&1 | tail -30

## Build the feasibility tests

In [ ]:
%%bash
set -e
cd /content/manifold
cp src/gpu/kernels/*.wgsl build/
for t in gpu_tests/fp64_feas.cpp gpu_tests/smoke.cpp gpu_tests/kernel12_feas.cpp gpu_tests/intersect_feas.cpp; do
  name=$(basename $t .cpp)
  g++ -std=c++17 -O3 -I include -I src -I build/include \
    "$t" \
    build/src/libmanifold.a \
    build/src/gpu/libmanifold_gpu.a \
    build/src/gpu/dawn/src/dawn/native/libwebgpu_dawn.a \
    -ltbb -lpthread -ldl -lvulkan \
    -o build/$name
  echo built: build/$name
done

## Test 1 — fp64 precision on Vulkan/NVIDIA

Target: add/sub ≤ 1 ULP, mul = 0 ULP, div ≤ 3 ULP.

In [ ]:
!cd /content/manifold/build && ./fp64_feas

## Test 2 — GPU Intersect() vs CPU

In [ ]:
!cd /content/manifold/build && ./intersect_feas

## Test 3 — end-to-end benchmark, GPU vs CPU

This is the key result. On Apple Silicon the GPU path was ~2× slower than CPU because
Dawn serializes Metal command encoding. Vulkan has no such constraint — if multi-device
and concurrent encoding both work, numbers should flip in GPU's favor.

In [ ]:
%%bash
cd /content/manifold/build
echo '=== GPU ==='
./smoke
echo
echo '=== CPU only ==='
MANIFOLD_GPU_DISABLE=1 ./smoke

## Test 4 — experimental GPU Kernel12 (fp64 math on GPU)

Starts with a tiny sphere union to prove the pipeline works on Vulkan without the TDR
hang we saw on Apple Silicon. If this completes, scale up with bigger `fn` values.

In [ ]:
%%bash
cd /content/manifold/build
for fn in 32 48 64 96; do
  echo "--- fn=$fn ---"
  timeout 120 env MANIFOLD_GPU_KERNEL=1 ./kernel12_feas $fn 2>&1 | tail -5 || echo "TIMEOUT at fn=$fn"
done